# Fine-tune TrOCR on `sample_1000` (Polish handwriting)

Fine-tunes `microsoft/trocr-base-handwritten` on the 1000 labeled line crops in
`output/sample_1000/` + `output/sample_1000_transcribed_v2.txt`, then evaluates on a held-out split.

**Before running:**
1. `Runtime > Change runtime type` and select a **GPU** (T4 is fine).
2. Upload your data to Google Drive — see Step 1 below.

In [2]:
!pip install -q "transformers>=4.35.0" "datasets>=2.14.0" accelerate

## Step 1 — Get the data into Colab

Upload these two files to your Google Drive (e.g. into `MyDrive/UMwTI/`):

- `sample_1000.zip` — a zip of the `output/sample_1000/` folder (1000 PNG line images)
- `sample_1000_transcribed_v2.txt` — the tab-separated transcription file

To create the zip locally (run once on your machine, not in Colab):

```python
import shutil
shutil.make_archive("sample_1000", "zip", "output", "sample_1000")
```

Then set `DATA_DIR` below to the Drive folder you uploaded the files to and run the cell.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import zipfile

DATA_DIR = Path("/content/drive/MyDrive/UMwTI")   # where you uploaded the files

WORK_DIR = Path("/content/data")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)        # clear stale files from previous runs
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = WORK_DIR / "combined_dataset"
TRANSCRIPTIONS_FILE = WORK_DIR / "combined_transcribed.txt"

if not IMAGES_DIR.exists():
    zip_path = DATA_DIR / "combined_dataset.zip"
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK_DIR)
    if not IMAGES_DIR.exists():
        IMAGES_DIR = WORK_DIR

if not TRANSCRIPTIONS_FILE.exists():
    shutil.copy(DATA_DIR / "combined_transcribed.txt", TRANSCRIPTIONS_FILE)

n_images = len(list(IMAGES_DIR.glob("*.png")))
print("Images dir:", IMAGES_DIR, "-", n_images, "images")
print("Transcriptions file:", TRANSCRIPTIONS_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting /content/drive/MyDrive/UMwTI/combined_dataset.zip ...
Images dir: /content/data/combined_dataset - 2585 images
Transcriptions file: /content/data/combined_transcribed.txt


## Step 2 — Config & imports

In [4]:
from __future__ import annotations

import inspect
import logging
import random
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from PIL import Image
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

TROCR_MODEL_BASE = "microsoft/trocr-base-handwritten"

# Where the fine-tuned model gets saved (on your Drive, so it survives the session)
OUTPUT_MODEL_DIR = Path("/content/drive/MyDrive/UMwTI/models/trocr_sample1000")

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 6
LEARNING_RATE = 5e-5
TEST_SIZE = 0.15
SEED = 42
MAX_TARGET_LENGTH = 128
PRINT_TEST_EXAMPLES = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## Step 3 — Load samples and build train/test splits

`sample_1000_transcribed_v2.txt` is tab-separated `image_name<TAB>text`.
Lines with an empty transcription (blank/illegible crops) are skipped.

In [5]:
def _levenshtein_distance(a: str, b: str) -> int:
    """Compute Levenshtein distance with dynamic programming."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(
                curr[j - 1] + 1,      # insertion
                prev[j] + 1,          # deletion
                prev[j - 1] + cost,   # substitution
            ))
        prev = curr
    return prev[-1]


def _character_error_rate(refs: list[str], preds: list[str]) -> float:
    total_dist = 0
    total_chars = 0
    for ref, pred in zip(refs, preds):
        total_dist += _levenshtein_distance(ref, pred)
        total_chars += max(1, len(ref))
    return total_dist / total_chars


def load_samples(images_dir: Path, transcription_file: Path) -> list[dict[str, str]]:
    """Load labeled samples and validate image existence."""
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    if not transcription_file.exists():
        raise FileNotFoundError(f"Transcription file not found: {transcription_file}")

    records: list[dict[str, str]] = []
    skipped_empty = 0
    with open(transcription_file, "r", encoding="utf-8") as f:
        for idx, raw_line in enumerate(f, start=1):
            line = raw_line.rstrip("\n")
            if not line.strip():
                continue
            if "\t" not in line:
                raise ValueError(f"Line {idx} in {transcription_file} is not tab-separated.")

            image_name, text = line.split("\t", 1)
            if not text.strip():
                skipped_empty += 1
                continue

            image_path = images_dir / image_name
            if not image_path.exists():
                raise FileNotFoundError(f"Missing image for line {idx}: {image_name}")

            records.append({"image_path": str(image_path), "text": text})

    if not records:
        raise ValueError("No valid samples were loaded.")

    logger.info(
        "Loaded %d labeled samples (%d skipped: empty transcription).",
        len(records), skipped_empty,
    )
    return records


def build_datasets(records: list[dict[str, str]], test_size: float, seed: int) -> tuple[Dataset, Dataset]:
    """Create train/test HF datasets."""
    dataset = Dataset.from_list(records)
    split = dataset.train_test_split(test_size=test_size, seed=seed, shuffle=True)
    train_ds = split["train"]
    test_ds = split["test"]
    logger.info("Split dataset: %d train / %d test", len(train_ds), len(test_ds))
    return train_ds, test_ds


records = load_samples(IMAGES_DIR, TRANSCRIPTIONS_FILE)
train_raw, test_raw = build_datasets(records, test_size=TEST_SIZE, seed=SEED)

## Step 4 — Load model & processor

In [6]:
processor = TrOCRProcessor.from_pretrained(TROCR_MODEL_BASE)
model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL_BASE)

# Ensure generation tokens are consistently configured.
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

if model.generation_config is not None:
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id

model.to(device)
print("Model loaded on", device)

# --- Aspect-ratio-preserving image prep -------------------------------------
# The line crops are extremely panoramic (median ~8.5:1, up to ~22:1). The
# default TrOCRProcessor resize squashes everything to 384x384 *ignoring aspect
# ratio*, which stretches strokes into blobs and destroys training. Instead we
# letterbox: scale keeping proportions to fit the square, then pad. We turn off
# the processor's own resize so it doesn't squash on top of our letterbox.
_img_size = processor.image_processor.size
try:
    # plain dict or SizeDict (mapping-like): {"height":384,"width":384} or {"shortest_edge":384}
    IMAGE_SIZE = int(_img_size["height"])
except (TypeError, KeyError):
    try:
        IMAGE_SIZE = int(_img_size["shortest_edge"])
    except (TypeError, KeyError):
        IMAGE_SIZE = int(getattr(_img_size, "height", None) or 384)

processor.image_processor.do_resize = False


def letterbox(img: "Image.Image", size: int = IMAGE_SIZE, bg: int = 255) -> "Image.Image":
    """Resize preserving aspect ratio to fit `size`x`size`, padding the rest."""
    img = img.convert("RGB")
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    img = img.resize((new_w, new_h), Image.BICUBIC)
    canvas = Image.new("RGB", (size, size), (bg, bg, bg))
    canvas.paste(img, (0, (size - new_h) // 2))  # left-aligned, vertically centered
    return canvas


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


## Step 5 — Preprocess datasets

In [7]:
def preprocess_dataset(ds: Dataset, processor: TrOCRProcessor, max_target_length: int) -> Dataset:
    """Tokenize the text labels only and keep the image path.

    We do NOT cache pixel tensors here: storing 3x384x384 floats per image blows
    the Arrow cache up to ~9 GB and crashes the Colab kernel at save time. Images
    are letterboxed + processed on the fly in the data collator instead (cheap,
    only one batch lives in memory at a time).
    """

    def _map_fn(examples: dict[str, list[str]]) -> dict[str, list[list[int]]]:
        labels = processor.tokenizer(
            examples["text"],
            padding="max_length",
            max_length=max_target_length,
            truncation=True,
        ).input_ids

        # Ignore pad tokens in the loss
        labels = [
            [token if token != processor.tokenizer.pad_token_id else -100 for token in seq]
            for seq in labels
        ]

        return {"labels": labels}

    return ds.map(
        _map_fn,
        batched=True,
        batch_size=64,
        remove_columns=[c for c in ds.column_names if c != "image_path"],
        desc="Tokenizing labels",
    )


@dataclass
class TrOCRDataCollator:
    """Letterbox + process images on the fly, then batch the tensors."""

    processor: TrOCRProcessor

    def __call__(self, features: list[dict]) -> dict[str, torch.Tensor]:
        images = [letterbox(Image.open(f["image_path"]).convert("RGB")) for f in features]
        pixel_values = self.processor(images=images, return_tensors="pt").pixel_values
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}


train_ds = preprocess_dataset(train_raw, processor, max_target_length=MAX_TARGET_LENGTH)
test_ds = preprocess_dataset(test_raw, processor, max_target_length=MAX_TARGET_LENGTH)


Tokenizing labels:   0%|          | 0/2197 [00:00<?, ? examples/s]

Tokenizing labels:   0%|          | 0/388 [00:00<?, ? examples/s]

## Step 6 — Metrics, training args & trainer

In [8]:
def build_compute_metrics(processor: TrOCRProcessor):
    """Build CER + exact-match metrics for Trainer."""

    def _compute_metrics(eval_preds):
        pred_ids, label_ids = eval_preds
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        # Newer transformers pads variable-length predictions across eval batches with -100;
        # replace those before decoding or the tokenizer raises an OverflowError.
        pred_ids = np.where(pred_ids != -100, pred_ids, processor.tokenizer.pad_token_id)

        # Replace ignore index to decode labels
        labels = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

        pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_texts = processor.batch_decode(labels, skip_special_tokens=True)

        pred_texts = [p.strip() for p in pred_texts]
        label_texts = [l.strip() for l in label_texts]

        cer = _character_error_rate(label_texts, pred_texts)
        exact_match = sum(p == l for p, l in zip(pred_texts, label_texts)) / max(1, len(label_texts))
        return {"cer": cer, "exact_match": exact_match}

    return _compute_metrics


def build_training_args(output_dir: Path, use_fp16: bool) -> Seq2SeqTrainingArguments:
    """Build Seq2SeqTrainingArguments across transformers API variants."""
    common_kwargs = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": EPOCHS,
        "save_strategy": "epoch",
        "logging_strategy": "steps",
        "logging_steps": 10,
        "predict_with_generate": True,
        "generation_max_length": MAX_TARGET_LENGTH,
        "remove_unused_columns": False,  # keep image_path for the on-the-fly collator
        "fp16": use_fp16,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "cer",
        "greater_is_better": False,
        "report_to": "none",
    }

    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    if "evaluation_strategy" in params:
        common_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in params:
        common_kwargs["eval_strategy"] = "epoch"
    else:
        logger.warning("No eval strategy arg found; falling back to no periodic evaluation.")

    return Seq2SeqTrainingArguments(**common_kwargs)


def build_trainer(
    model: VisionEncoderDecoderModel,
    training_args: Seq2SeqTrainingArguments,
    train_ds: Dataset,
    test_ds: Dataset,
    processor: TrOCRProcessor,
) -> Seq2SeqTrainer:
    """Build Seq2SeqTrainer across transformers API variants."""
    kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "eval_dataset": test_ds,
        "data_collator": TrOCRDataCollator(processor),
        "compute_metrics": build_compute_metrics(processor),
    }

    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "tokenizer" in params:
        kwargs["tokenizer"] = processor.tokenizer
    elif "processing_class" in params:
        kwargs["processing_class"] = processor
    else:
        logger.warning("No tokenizer/processing_class argument found in Seq2SeqTrainer.")

    return Seq2SeqTrainer(**kwargs)


# Checkpoints are written locally (fast disk); the final model is saved to Drive in Step 8.
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

use_fp16 = torch.cuda.is_available()
training_args = build_training_args(CHECKPOINT_DIR, use_fp16)
trainer = build_trainer(model, training_args, train_ds, test_ds, processor)

## Step 7 — Train

In [9]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Cer,Exact Match
1,3.749823,3.944064,0.921763,0.000000
2,3.273517,3.587876,0.948565,0.002577
3,3.234879,3.469439,0.848565,0.000000
4,2.933201,3.448640,0.875437,0.002577
5,2.912747,3.384958,0.877887,0.002577
6,2.484263,3.440161,0.924633,0.002577
7,1.991218,3.567904,0.908607,0.002577
8,1.568206,3.711090,0.907908,0.002577
9,1.273341,3.790376,0.910357,0.002577
10,0.989386,3.851245,0.905178,0.002577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.layers.0.attention.q_proj.weight', 'encoder.layers.0.attention.k_proj.weight', 'encoder.layers.0.attention.v_proj.weight', 'encoder.layers.0.attention.o_proj.weight', 'encoder.layers.0.attention.o_proj.bias', 'encoder.layers.0.layernorm_before.weight', 'encoder.layers.0.layernorm_before.bias', 'encoder.layers.0.layernorm_after.weight', 'encoder.layers.0.layernorm_after.bias', 'encoder.layers.0.mlp.fc1.weight', 'encoder.layers.0.mlp.fc1.bias', 'encoder.layers.0.mlp.fc2.weight', 'encoder.layers.0.mlp.fc2.bias', 'encoder.layers.1.attention.q_proj.weight', 'encoder.layers.1.attention.k_proj.weight', 'encoder.layers.1.attention.v_proj.weight', 'encoder.layers.1.attention.o_proj.weight', 'encoder.layers.1.attention.o_proj.bias', 'encoder.layers.1.layernorm_before.weight', 'encoder.layers.1.layernorm_before.bias', 'encoder.layers.1.layernorm_after.weight', 'encoder.layers.1.layernorm_after.bias', 'encoder.layers.

TrainOutput(global_step=3670, training_loss=2.5160139595780127, metrics={'train_runtime': 3109.7587, 'train_samples_per_second': 7.065, 'train_steps_per_second': 1.18, 'total_flos': 1.6439825646181417e+19, 'train_loss': 2.5160139595780127, 'epoch': 10.0})

In [10]:
eval_metrics = trainer.evaluate()
print("Final metrics:", eval_metrics)

Training Loss,Validation Loss,Epoch,Cer,Exact Match
0.989386,3.518713,10,0.865220,0.002577


Final metrics: {'eval_loss': 3.5187129974365234, 'eval_cer': 0.865220433869839, 'eval_exact_match': 0.002577319587628866}


## Step 8 — Save the fine-tuned model to Drive

In [11]:
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_MODEL_DIR))
processor.save_pretrained(str(OUTPUT_MODEL_DIR))
print("Saved model to", OUTPUT_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /content/drive/MyDrive/UMwTI/models/trocr_sample1000


## Step 9 — Qualitative test on held-out examples

In [12]:
def run_qualitative_test(
    model: VisionEncoderDecoderModel,
    processor: TrOCRProcessor,
    test_records: list[dict[str, str]],
    num_examples: int,
) -> None:
    """Print sample predictions from the test split."""
    if not test_records:
        print("No test samples available for qualitative test.")
        return

    model.eval()
    device = model.device

    subset = test_records[: min(num_examples, len(test_records))]
    for i, record in enumerate(subset, start=1):
        image = letterbox(Image.open(record["image_path"]).convert("RGB"))
        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values, max_new_tokens=128)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        truth = record["text"].strip()
        print(f"[{i}] GT: {truth}")
        print(f"    PR: {pred}")


test_records = [test_raw[i] for i in range(len(test_raw))]
run_qualitative_test(trainer.model, processor, test_records, PRINT_TEST_EXAMPLES)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] GT: połączenia z Tobą, dodaję zarazem, abyś
    PR: i bardzo cieszę, że nie chwile
[2] GT: pyta: „jakże tam Twoje stosunki
    PR: Tak mała


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3] GT: pozostaną również tylko wspólną tajemnicą; jeszczem ci to
    PR: i bardzo cieszę, że nie chwile


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[7] GT: Hej Mój kraj droższy, Serdecznie
    PR: Dziś nie mnie mnie mnie nie chwilej


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[8] GT: i przy kąpieli będzie zawsze kucharka, a gdy wchodzisz
    PR: bądą bardzo cieszę bardzo ciędzie


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[9] GT: staje, gdy walcząc z przeszkodami walczy sam do zupełnego zwy-
    PR: Piszą, że nie mnie nie mniej nie chwilę
[10] GT: przepełniam Cię jestem. Jesteś mi tak strasznie droga.
    PR: wielką, że nie mnie nie mniej nie chwilej
